# B.12 — v2 LoRA per-row eval + leakage-clean slice  
**GPU: T4** &nbsp;·&nbsp; **Cost: ~1 Colab unit** &nbsp;·&nbsp; **Runtime: ~7 min**

Loads `ujjwalpardeshi/chakravyuh-analyzer-lora-v2` over Qwen2.5-7B-Instruct (4-bit), scores all 174 bench scenarios using the exact training prompt, joins with the semantic-leakage audit, and emits the leakage-clean OOD number (the answer to Q&A 3b).

Outputs three artifacts:
- `logs/eval_v2_per_row.jsonl` &nbsp;— per-scenario {id, score, predicted, ground_truth, leakage_cosine}
- `logs/eval_v2_reproduce.json` &nbsp;— aggregate metrics + leakage-clean slices
- `docs/leakage_clean_eval.md` &nbsp;— markdown report ready to commit

## Run order
1. `Runtime → Disconnect and delete runtime`, then connect with **GPU T4**.
2. **Run cells in order (NOT "Run all"):** Cell 1 (GPU check), Cell 2 (clone), Cell 3 (install + auto-restart).
3. Kernel will die at end of Cell 3 — that's intentional. Wait ~10 sec for auto-reconnect.
4. After reconnect: re-run Cell 2 (clone, idempotent) to reset variables, then **SKIP Cell 3** and run Cells 4 onwards.

In [ ]:
# CELL 1: GPU check + assert T4
import subprocess, sys
print('Python:', sys.version)
out = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode().strip()
print('GPU:', out)
assert 'T4' in out or 'L4' in out or 'A100' in out, f'Need GPU runtime; got: {out}'

In [ ]:
# CELL 2: Clone repo (idempotent — safe to re-run after kernel restart)
import os
from pathlib import Path
REPO_URL = 'https://github.com/UjjwalPardeshi/Chakravyuh'
REPO_DIR = '/content/Chakravyuh'
if not Path(REPO_DIR).exists():
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --rebase --autostash
%cd {REPO_DIR}
!git rev-parse HEAD

In [ ]:
# CELL 3: One-shot install + KERNEL RESTART. Self-contained — does not depend on Cell 2.
import os, time
REPO_DIR = '/content/Chakravyuh'
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 https://github.com/UjjwalPardeshi/Chakravyuh {REPO_DIR}
os.chdir(REPO_DIR)

# 3a. Wipe Colab's broken torch+torchvision combo
!pip uninstall -y -q torch torchvision torchaudio 2>&1 | tail -2

# 3b. Matched cu121 torch+torchvision pair from PyTorch official wheels
!pip install -q --no-cache-dir --index-url https://download.pytorch.org/whl/cu121 \
    'torch==2.5.1+cu121' 'torchvision==0.20.1+cu121' 2>&1 | tail -3

# 3c. Chakravyuh + ML deps with --no-deps (so nothing pulls torchvision back)
!pip install -q -e {REPO_DIR} --no-deps 2>&1 | tail -2

# 3c1. Pinned ML stack — tokenizers MUST be <0.21 for transformers 4.46.3
!pip install -q --no-deps \
    'pydantic>=2.6' 'python-dotenv>=1.0' 'jsonlines>=4.0' \
    'openenv-core>=0.2.3,<0.3' 'fastapi>=0.115' 'uvicorn>=0.30' 'tqdm' \
    'transformers==4.46.3' 'peft>=0.14,<0.20' 'accelerate==1.0.1' 'bitsandbytes==0.44.1' \
    'tokenizers>=0.20,<0.21' 'huggingface-hub>=0.24,<0.27' \
    'sentencepiece' 'safetensors' \
    2>&1 | tail -3

# 3c2. fastmcp + mcp (transitive deps of openenv-core that --no-deps skipped)
!pip install -q 'fastmcp>=0.4' 'mcp>=1.0' 'httpx' 'anyio' 2>&1 | tail -3

# 3d. Verify torch+torchvision are matched + transformers is loadable
!python -c "import torch, torchvision; print(f'torch: {torch.__version__} | torchvision: {torchvision.__version__}')"
!python -c "from transformers.cache_utils import SlidingWindowCache; print('SlidingWindowCache: OK')"
!python -c "import openenv; print('openenv: import OK')" 2>&1 | tail -2

# 3e. Kill kernel — the new torch must be loaded fresh on auto-reconnect
print('\n' + '=' * 60)
print('SETUP COMPLETE. Kernel restarting in 2s...')
print('After restart: re-run Cell 2 (clone), SKIP Cell 3, run Cells 4 onwards.')
print('=' * 60)
time.sleep(2)
os.kill(os.getpid(), 9)


In [ ]:
# CELL 4: Load v2 LoRA — with defensive adapter_config sanitization.
import sys, os, json
sys.path.insert(0, REPO_DIR)

from huggingface_hub import snapshot_download
from peft import LoraConfig
from chakravyuh_env.agents.llm_analyzer import LLMAnalyzer

BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
LORA_REPO  = 'ujjwalpardeshi/chakravyuh-analyzer-lora-v2'
LOCAL_LORA = '/content/lora_v2_local'

# 4a. Snapshot LoRA locally + sanitize adapter_config.json
if not os.path.exists(LOCAL_LORA):
    snapshot_download(repo_id=LORA_REPO, local_dir=LOCAL_LORA, local_dir_use_symlinks=False)
cfg_path = os.path.join(LOCAL_LORA, 'adapter_config.json')
with open(cfg_path) as f:
    cfg = json.load(f)
valid_keys = set(LoraConfig.__dataclass_fields__.keys()) | {'peft_type', 'task_type', 'auto_mapping', 'base_model_name_or_path', 'revision', 'inference_mode'}
removed = [k for k in list(cfg.keys()) if k not in valid_keys]
if removed:
    print(f'Stripping unknown adapter_config keys: {removed}')
    for k in removed: cfg.pop(k, None)
    with open(cfg_path, 'w') as f: json.dump(cfg, f, indent=2)
else:
    print('adapter_config.json: all keys recognized.')

analyzer = LLMAnalyzer(
    model_name=BASE_MODEL,
    lora_path=LOCAL_LORA,
    use_unsloth=False,
    load_in_4bit=True,
    max_new_tokens=200,
    temperature=0.0,
    flag_threshold=0.55,
    seed=42,
)
analyzer.load()
print('Analyzer ready. Threshold =', analyzer.flag_threshold)


In [ ]:
# CELL 5: Sanity check — known scam should score >= 0.7, known benign <= 0.3
smoke_scam = 'URGENT: Your SBI account will be blocked today. Share OTP 234567 to verify KYC.'
smoke_benign = 'Your OTP for HDFC login is 563182. Valid 5 min. Do NOT share with anyone.'
scam_s = analyzer.score_text(smoke_scam)
benign_s = analyzer.score_text(smoke_benign)
print(f'Scam test  : {scam_s:.3f}  (expected >= 0.7)')
print(f'Benign test: {benign_s:.3f}  (expected <= 0.3)')
assert scam_s >= 0.5, 'LoRA may not have loaded correctly — scam scored too low'
assert benign_s <= 0.5, 'LoRA may not have loaded correctly — benign scored too high'
print('Sanity check PASSED.')

In [ ]:
# CELL 6: Score every bench scenario
import json
from pathlib import Path
from tqdm.auto import tqdm
import time

BENCH = Path(REPO_DIR) / 'data' / 'chakravyuh-bench-v0' / 'scenarios.jsonl'
LEAK = Path(REPO_DIR) / 'logs' / 'semantic_leakage_audit.json'
with BENCH.open() as f:
    scenarios = [json.loads(l) for l in f if l.strip()]
with LEAK.open() as f:
    leak = json.load(f)
leak_by_id = {r['scenario_id']: r for r in leak['per_scenario']}
print(f'Bench: {len(scenarios)} scenarios | Leakage data: {len(leak_by_id)} rows')

THR = analyzer.flag_threshold
per_row, errors = [], []
t0 = time.time()
for s in tqdm(scenarios, desc='scoring'):
    text = ' '.join(m['text'] for m in s['attack_sequence'] if m['sender'] == 'scammer')
    try:
        score = float(analyzer.score_text(text))
    except Exception as e:
        errors.append({'scenario_id': s['id'], 'error': str(e)})
        continue
    leak_row = leak_by_id.get(s['id'], {})
    per_row.append({
        'scenario_id': s['id'],
        'score': score,
        'predicted_flag': bool(score >= THR),
        'ground_truth': bool(s['ground_truth']['is_scam']),
        'correct': (score >= THR) == s['ground_truth']['is_scam'],
        'difficulty': s['ground_truth']['difficulty'],
        'category': s['ground_truth']['category'],
        'leakage_cosine': leak_row.get('max_cosine_to_training'),
        'nearest_template_id': leak_row.get('nearest_template_id'),
    })
elapsed = time.time() - t0
print(f'\nScored {len(per_row)} / {len(scenarios)} in {elapsed/60:.1f} min ({elapsed/max(len(per_row),1):.2f} sec/row).')
if errors:
    print(f'Skipped {len(errors)} scenarios:', errors[:3])

In [ ]:
# CELL 7: Aggregate + per-difficulty + leakage-clean slices
from collections import defaultdict

def aggregate(rows):
    n = len(rows)
    if not n: return {'n': 0}
    tp = sum(1 for r in rows if r['ground_truth'] and r['predicted_flag'])
    fn = sum(1 for r in rows if r['ground_truth'] and not r['predicted_flag'])
    fp = sum(1 for r in rows if not r['ground_truth'] and r['predicted_flag'])
    tn = sum(1 for r in rows if not r['ground_truth'] and not r['predicted_flag'])
    scam, benign = tp + fn, fp + tn
    det = tp / scam if scam else 0.0
    fpr = fp / benign if benign else 0.0
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    f1 = (2 * prec * det / (prec + det)) if (prec + det) else 0.0
    return {'n': n, 'n_scam': scam, 'n_benign': benign, 'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn,
            'detection': det, 'fpr': fpr, 'precision': prec, 'f1': f1, 'accuracy': (tp + tn) / n}

overall = aggregate(per_row)
by_diff = defaultdict(list)
for r in per_row: by_diff[r['difficulty']].append(r)
by_diff = {d: aggregate(rs) for d, rs in by_diff.items()}

LEAK_THRESHOLDS = [0.50, 0.70, 0.85, 0.95]
slices = {thr: aggregate([r for r in per_row if r['leakage_cosine'] is not None and r['leakage_cosine'] < thr]) for thr in LEAK_THRESHOLDS}

print('=== Overall (full bench, in-distribution) ===')
print(f"detection: {overall['detection']:.1%} | FPR: {overall['fpr']:.1%} | F1: {overall['f1']:.4f}")
print('\n=== Per-difficulty ===')
for d in ('easy', 'medium', 'hard', 'novel'):
    if d in by_diff:
        m = by_diff[d]
        print(f"  {d:8s} n={m['n']:3d}  det={m['detection']:.1%}  fpr={m['fpr']:.1%}")
print('\n=== Leakage-clean slices ===')
for thr in LEAK_THRESHOLDS:
    m = slices[thr]
    if m['n'] == 0: continue
    print(f"  cosine<{thr:.2f}: n={m['n']:3d} (scam={m['n_scam']}, benign={m['n_benign']})  det={m['detection']:.1%}  fpr={m['fpr']:.1%}")
honest = slices[0.70]
if honest['n'] > 0:
    print(f"\n>>> HONEST OOD HEADLINE (cosine<0.70, n={honest['n']}): det={honest['detection']:.1%}, FPR={honest['fpr']:.1%}, F1={honest['f1']:.4f}")

In [ ]:
# CELL 8: Save artifacts (3 files)
from pathlib import Path
LOGS = Path(REPO_DIR) / 'logs'
DOCS = Path(REPO_DIR) / 'docs'
LOGS.mkdir(exist_ok=True); DOCS.mkdir(exist_ok=True)

per_row_path = LOGS / 'eval_v2_per_row.jsonl'
with per_row_path.open('w') as f:
    for r in per_row: f.write(json.dumps(r) + '\n')

agg_path = LOGS / 'eval_v2_reproduce.json'
with agg_path.open('w') as f:
    json.dump({
        'lora_v2': {
            'detection': overall['detection'], 'fpr': overall['fpr'],
            'precision': overall['precision'], 'f1': overall['f1'], 'n': overall['n'],
            'per_difficulty': {d: {'n': m['n'], 'detection_rate': m['detection']} for d, m in by_diff.items()},
            'threshold': THR,
        },
        'leakage_clean_slices': {f'cosine_lt_{thr:.2f}': slices[thr] for thr in LEAK_THRESHOLDS},
        'meta': {'base_model': BASE_MODEL, 'lora_adapter': LORA_ADAPTER, 'threshold': THR,
                 'temperature': 0.0, 'seed': 42, 'n_attempted': len(scenarios), 'n_scored': len(per_row), 'errors': len(errors)},
    }, f, indent=2)

md_lines = [
    '# v2 LoRA — leakage-clean evaluation\n',
    '_Generated by `notebooks/T4_b12_per_row_eval.ipynb`. Source: `logs/eval_v2_per_row.jsonl`._\n',
    f'\n**Base:** `{BASE_MODEL}`  ',
    f'\n**LoRA:** `{LORA_ADAPTER}`  ',
    f'\n**Threshold:** {THR}  &nbsp;**Seed:** 42  &nbsp;**Temp:** 0.0\n',
    '\n## Headline numbers\n',
    '| Slice | n | n_scam | n_benign | Detection | FPR | F1 |',
    '|---|---:|---:|---:|---:|---:|---:|',
    f"| **Full bench (in-distribution)** | {overall['n']} | {overall['n_scam']} | {overall['n_benign']} | {overall['detection']:.1%} | {overall['fpr']:.1%} | {overall['f1']:.4f} |",
]
for thr in LEAK_THRESHOLDS:
    m = slices[thr]
    if m['n'] == 0: continue
    label = f'Leakage-clean (cosine < {thr:.2f})'
    if abs(thr - 0.70) < 1e-9: label = f'**{label}** ⭐ honest OOD'
    md_lines.append(f"| {label} | {m['n']} | {m['n_scam']} | {m['n_benign']} | {m['detection']:.1%} | {m['fpr']:.1%} | {m['f1']:.4f} |")
md_lines += ['\n## Per-difficulty (full bench)', '| Difficulty | n | Detection |', '|---|---:|---:|']
for d in ('easy', 'medium', 'hard', 'novel'):
    if d in by_diff: md_lines.append(f"| {d} | {by_diff[d]['n']} | {by_diff[d]['detection']:.1%} |")
md_path = DOCS / 'leakage_clean_eval.md'
md_path.write_text('\n'.join(md_lines) + '\n')

print('Saved:')
for p in (per_row_path, agg_path, md_path): print(' -', p)

In [ ]:
# CELL 9: Download artifacts
try:
    from google.colab import files
    for p in (per_row_path, agg_path, md_path): files.download(str(p))
except ImportError:
    print('Not in Colab — files at:'); [print(' ', p) for p in (per_row_path, agg_path, md_path)]

## Next steps

Drop the 3 downloaded files into your local repo, commit + push:
```bash
mv ~/Downloads/eval_v2_per_row.jsonl logs/
mv ~/Downloads/eval_v2_reproduce.json logs/
mv ~/Downloads/leakage_clean_eval.md docs/
git add logs/eval_v2_per_row.jsonl logs/eval_v2_reproduce.json docs/leakage_clean_eval.md
git commit -m "feat(eval): B.12 per-row v2 logits + leakage-clean slice"
git push
```

**Update `docs/limitations.md`** with the leakage-clean detection / FPR / F1 numbers (replace the placeholder *"v3 work — flagged below"*).  
**Update `docs/Q_AND_A_REHEARSAL.md` Q3b** with the measured number for the leakage follow-up question.